<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/memory/http_backed_memory_block.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HTTP-Backed Memory Block

This notebook shows how to build a custom `BaseMemoryBlock` that persists flushed chat history to an external HTTP service and retrieves relevant context on demand.

The pattern is useful when memory must live outside the agent process — for example, a team-owned memory API, an existing user-profile service, or a shared session store.

We use only the Python standard library (`urllib`, `http.server`) so no extra dependencies are required.

In [ ]:
%pip install llama-index-core

## Mock HTTP memory service

For demonstration, we spin up a tiny in-process HTTP server that stores memory batches in a dictionary. Replace this with your production endpoint in real deployments.

In [ ]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from typing import Dict, List

STORE: Dict[str, List[str]] = {}


class MemoryHandler(BaseHTTPRequestHandler):
    def log_message(self, format: str, *args) -> None:
        return  # keep notebook output clean

    def do_GET(self) -> None:
        session_id = self.path.strip("/").split("?")[0]
        memory = "\n".join(STORE.get(session_id, []))
        body = json.dumps({"memory": memory}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)

    def do_POST(self) -> None:
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length).decode())
        session_id = payload["session_id"]
        batch = "\n".join(
            f"<{m['role']}>{m['content']}</{m['role']}>" for m in payload["messages"]
        )
        STORE.setdefault(session_id, []).append(batch)
        self.send_response(201)
        self.end_headers()


server = HTTPServer(("127.0.0.1", 8765), MemoryHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print("Mock memory service listening on http://127.0.0.1:8765")

## Define the HTTP-backed memory block

In [ ]:
import asyncio
import json
from typing import Any, List, Optional
from urllib import request

from llama_index.core.llms import ChatMessage
from llama_index.core.memory import BaseMemoryBlock, Memory


class HttpBackedMemoryBlock(BaseMemoryBlock[str]):
    endpoint_url: str = "http://127.0.0.1:8765"
    session_id: str = "demo-session"
    timeout_seconds: float = 5.0

    async def _aget(
        self, messages: Optional[List[ChatMessage]] = None, **block_kwargs: Any
    ) -> str:
        url = f"{self.endpoint_url}/{self.session_id}"

        def _fetch() -> str:
            with request.urlopen(url, timeout=self.timeout_seconds) as resp:
                payload = json.loads(resp.read().decode())
                return payload.get("memory", "")

        return await asyncio.to_thread(_fetch)

    async def _aput(self, messages: List[ChatMessage]) -> None:
        body = json.dumps(
            {
                "session_id": self.session_id,
                "messages": [
                    {"role": m.role, "content": m.content} for m in messages
                ],
            }
        ).encode()

        def _post() -> None:
            req = request.Request(
                self.endpoint_url,
                data=body,
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with request.urlopen(req, timeout=self.timeout_seconds):
                pass

        await asyncio.to_thread(_post)

    async def atruncate(
        self, content: str, tokens_to_truncate: int
    ) -> Optional[str]:
        lines = content.splitlines()
        drop = max(1, tokens_to_truncate // 4)
        return "\n".join(lines[drop:]) if len(lines) > drop else ""

## Wire memory block into `Memory`

When short-term memory exceeds its token budget, flushed messages are sent to the HTTP service via `_aput()`. On retrieval, `_aget()` fetches the archived context.

In [ ]:
memory = Memory.from_defaults(
    session_id="demo-session",
    token_limit=80,
    token_flush_size=20,
    chat_history_token_ratio=0.5,
    memory_blocks=[HttpBackedMemoryBlock(session_id="demo-session")],
)

memory.put_messages(
    [
        ChatMessage(role="user", content="I prefer responses in bullet points."),
        ChatMessage(role="assistant", content="Got it — I'll use bullet points."),
        ChatMessage(role="user", content="My project codename is Northstar."),
        ChatMessage(role="assistant", content="Noted: project Northstar."),
        ChatMessage(role="user", content="What format do I prefer?"),
    ]
)

chat_history = memory.get()
print(chat_history[0].content)

You should see archived messages from the HTTP store merged into the retrieved memory. In production, point `endpoint_url` at your team's memory API and scope records with `session_id`.